# Figure 4: External validation and clinical relevance of the learned embeddings

Reads the TCGA run outputs from `Runs/TCGA/output/cpm/` (sample embeddings, predictions), the TCGA expression matrix the model was trained on (E2M with `Runs/TCGA/config/e2m.yaml`), MC3 gene-level labels from `Runs/TCGA/data/mutations/`, TCGA clinical data from `data/GDC-PANCAN.TCGA_phenotype.tsv.gz`, and the external-validation outputs from `Runs/External/output/`. Panels and source tables are written to `Figures/Figure4/output/`.

Run the notebook part by part; each part writes its source tables under `output/source_data/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
FIGURES_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = FIGURES_DIR.parent
for path in (NOTEBOOK_DIR, FIGURES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import figure4_helpers as f4

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

## 0. Inputs and outputs

In [ ]:
config = f4.Figure4Config.from_project_root(PROJECT_ROOT)
config.ensure_inputs()
config.ensure_outputs()

path_table = config.as_frame()
dependency_status = f4.projection_dependency_status()

f4.print_and_save(path_table, config, "figure4_active_paths.csv")
f4.print_and_save(dependency_status, config, "figure4_projection_dependency_status.csv")

display(path_table)
display(dependency_status)

## 1. TCGA Embedding Panels

Loads the TCGA sample embeddings and the model's expression matrix, clusters embedding, expression and mutation-profile representations, renders UMAP/KM panels, and saves cluster/KM summaries.

In [ ]:
TCGA_CANCERS = None  # None = every cancer in TCGA_PARAMS; or e.g. ["BRCA", "LUAD", "UCEC"]
RUN_TCGA_PLOTS = True
INCLUDE_MUTATION_PROFILE = True

TCGA_UMAP_FIGSIZE = (2.2, 2.0)
TCGA_KM_FIGSIZE = (2.7, 2.3)
TCGA_KM_MIN_GROUP_SIZE = 10
TCGA_SHOW_UMAP_LEGEND = False
TCGA_MAX_TIME_BY_CANCER = {"UCEC": 3000}

# Tunable TCGA representation parameters. Mutation PCA starts from the embedding parameters.
TCGA_PARAMS = {
    "BLCA": {"expr_nn": 4, "expr_leiden_res": 0.2, "emb_nn": 4, "emb_leiden_res": 0.2, "mut_nn": 4, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 3, "mut_clusters": 3, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "BRCA": {"expr_nn": 4, "expr_leiden_res": 0.20, "emb_nn": 4, "emb_leiden_res": 0.2, "mut_nn": 4, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 3, "emb_clusters": 8, "mut_clusters": 8, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "COAD": {"expr_nn": 5, "expr_leiden_res": 0.15, "emb_nn": 5, "emb_leiden_res": 0.15, "mut_nn": 5, "mut_leiden_res": 0.15, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "KIRC": {"expr_nn": 3, "expr_leiden_res": 0.30, "emb_nn": 3, "emb_leiden_res": 0.30, "mut_nn": 3, "mut_leiden_res": 0.30, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 10, "mut_clusters": 10, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "KIRP": {"expr_nn": 3, "expr_leiden_res": 0.2, "emb_nn": 3, "emb_leiden_res": 0.2, "mut_nn": 3, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 10, "mut_clusters": 10, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "LGG": {"expr_nn": 5, "expr_leiden_res": 0.2, "emb_nn": 5, "emb_leiden_res": 0.2, "mut_nn": 5, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 10, "mut_clusters": 10, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "LIHC": {"expr_nn": 3, "expr_leiden_res": 0.15, "emb_nn": 3, "emb_leiden_res": 0.15, "mut_nn": 3, "mut_leiden_res": 0.15, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 8, "mut_clusters": 8, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "LUAD": {"expr_nn": 6, "expr_leiden_res": 0.40, "emb_nn": 6, "emb_leiden_res": 0.4, "mut_nn": 6, "mut_leiden_res": 0.4, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "PAAD": {"expr_nn": 5, "expr_leiden_res": 0.2, "emb_nn": 5, "emb_leiden_res": 0.2, "mut_nn": 5, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "PCPG": {"expr_nn": 4, "expr_leiden_res": 0.35, "emb_nn": 4, "emb_leiden_res": 0.35, "mut_nn": 4, "mut_leiden_res": 0.35, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 4, "mut_clusters": 4, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "PRAD": {"expr_nn": 10, "expr_leiden_res": 0.5, "emb_nn": 10, "emb_leiden_res": 0.5, "mut_nn": 10, "mut_leiden_res": 0.5, "mut_pca_components": 256, "expr_clusters": 2, "emb_clusters": 6, "mut_clusters": 6, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "SARC": {"expr_nn": 8, "expr_leiden_res": 0.4, "emb_nn": 8, "emb_leiden_res": 0.4, "mut_nn": 8, "mut_leiden_res": 0.4, "mut_pca_components": 256, "expr_clusters": 3, "emb_clusters": 4, "mut_clusters": 4, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "SKCM": {"expr_nn": 8, "expr_leiden_res": 0.4, "emb_nn": 8, "emb_leiden_res": 0.4, "mut_nn": 8, "mut_leiden_res": 0.4, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "STAD": {"expr_nn": 5, "expr_leiden_res": 0.2, "emb_nn": 5, "emb_leiden_res": 0.20, "mut_nn": 5, "mut_leiden_res": 0.20, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "TGCT": {"expr_nn": 5, "expr_leiden_res": 0.2, "emb_nn": 5, "emb_leiden_res": 0.2, "mut_nn": 5, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 3, "emb_clusters": 6, "mut_clusters": 6, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "THCA": {"expr_nn": 5, "expr_leiden_res": 0.2, "emb_nn": 5, "emb_leiden_res": 0.2, "mut_nn": 5, "mut_leiden_res": 0.2, "mut_pca_components": 256, "expr_clusters": 3, "emb_clusters": 3, "mut_clusters": 3, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "THYM": {"expr_nn": 4, "expr_leiden_res": 0.20, "emb_nn": 4, "emb_leiden_res": 0.20, "mut_nn": 4, "mut_leiden_res": 0.20, "mut_pca_components": 256, "expr_clusters": 5, "emb_clusters": 3, "mut_clusters": 3, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
    "UCEC": {"expr_nn": 4, "expr_leiden_res": 0.25, "emb_nn": 4, "emb_leiden_res": 0.20, "mut_nn": 4, "mut_leiden_res": 0.30, "mut_pca_components": 256, "expr_clusters": 4, "emb_clusters": 2, "mut_clusters": 2, "expr_cluster_method": "leiden", "emb_cluster_method": "leiden", "mut_cluster_method": "leiden", "mut_min_dist": 0.3},
}

tcga_part = f4.run_tcga_part(
    config,
    cancers=TCGA_CANCERS,
    params=TCGA_PARAMS,
    plot=RUN_TCGA_PLOTS,
    include_mutation=INCLUDE_MUTATION_PROFILE,
    km_min_group_size=TCGA_KM_MIN_GROUP_SIZE,
    umap_figsize=TCGA_UMAP_FIGSIZE,
    km_figsize=TCGA_KM_FIGSIZE,
    show_umap_legend=TCGA_SHOW_UMAP_LEGEND,
    max_time_by_cancer=TCGA_MAX_TIME_BY_CANCER,
)

display(tcga_part["bundle_summary"])
display(tcga_part["km_summary"])
display(tcga_part["plot_summary"])

In [ ]:
# Text check: UCEC is the key TCGA survival example in the current Figure 4 draft.
ucec_clusters = tcga_part["bundle_summary"].query("cohort == 'UCEC'")
ucec_km = tcga_part["km_summary"].query("cohort == 'UCEC'") if not tcga_part["km_summary"].empty else pd.DataFrame()
display(ucec_clusters)
display(ucec_km)

## 2. External Mutation-Prediction Performance

Loads all current external validation bundles, writes an overview table, cohort-level top normalized-AUPRC genes, and CPTAC/CMI site-level top genes.

In [ ]:
EXTERNAL_TOP_N = 10
EXTERNAL_MIN_POSITIVE = 5
EXTERNAL_SCORE_COL = "normalized_auprc"
CPTAC_SITE_MIN_SAMPLES = 40
CPTAC_SITE_MIN_POSITIVE = 10

external_prediction = f4.summarize_external_prediction_part(
    config,
    top_n=EXTERNAL_TOP_N,
    min_positive=EXTERNAL_MIN_POSITIVE,
    cptac_min_site_samples=CPTAC_SITE_MIN_SAMPLES,
    cptac_min_positive=CPTAC_SITE_MIN_POSITIVE,
    score_col=EXTERNAL_SCORE_COL,
)

display(external_prediction["overview"])
display(external_prediction["top_metrics"])
display(external_prediction["cptac_site_top"])


In [ ]:
# Text check: compact top-gene view by cohort.
top3_by_cohort = (
    external_prediction["top_metrics"]
    .query("rank <= 3")
    .sort_values(["cohort", "rank"])
)
display(top3_by_cohort)

## 2b. External AUPRC Tile Plots And Integration UMAPs

Render the manuscript tile plots (cohort-level + CPTAC/CMI site-stratified) and the before/after batch-integration UMAPs directly from the bundles loaded in Part 2. Tiles use the prevalence-normalized AUPRC on a fixed 0-1 scale; integration coordinates are precomputed in each bundle and are not re-embedded here.

In [ ]:
# AUPRC tile plots: one row per cohort (top genes by normalized AUPRC) + a shared 0-1 legend,
# plus CPTAC/CMI site tiles (combined stacked figure AND one PDF per primary site).
# PDFs -> Figures/Figure4/<cohort>/<prefix>_top{N}_normalized_auprc_gene_tiles.pdf
#         Figures/Figure4/auprc_tile_legend_0_1.pdf
#         Figures/Figure4/cptac_cmi/cptac_site_top{N}_normalized_auprc_gene_tiles.pdf   (combined)
#         Figures/Figure4/cptac_cmi/cptac_<site>_top{N}_normalized_auprc_gene_tiles.pdf (per site)
external_tiles = f4.plot_external_auprc_tiles(
    external_prediction, config,
    top_n=EXTERNAL_TOP_N, min_positive=EXTERNAL_MIN_POSITIVE, score_col=EXTERNAL_SCORE_COL,
)
cptac_site_tiles = f4.plot_cptac_site_auprc_tiles(
    external_prediction, config,
    top_n=EXTERNAL_TOP_N, min_site_samples=CPTAC_SITE_MIN_SAMPLES,
    min_positive=CPTAC_SITE_MIN_POSITIVE, score_col=EXTERNAL_SCORE_COL,
)
cptac_per_site_tiles = f4.plot_cptac_per_site_auprc_tiles(
    external_prediction, config,
    top_n=EXTERNAL_TOP_N, min_site_samples=CPTAC_SITE_MIN_SAMPLES,
    min_positive=CPTAC_SITE_MIN_POSITIVE, score_col=EXTERNAL_SCORE_COL,
)
print("Per-site tiles written for:", [f4._cptac_site_label(s) for s in cptac_per_site_tiles])
display(external_tiles)

In [ ]:
# Integration UMAPs: TCGA background vs external foreground, before and after normalization.
# Coordinates are precomputed in each bundle (integration/*.csv); nothing is re-embedded here.
# CPTAC/CMI foreground is colored by sequencing batch; other cohorts by dataset.
# PDFs -> Figures/Figure4/<cohort>/<prefix>_integration_{before,after}.pdf
integration_results = f4.plot_all_external_integration(config, show_legend=True)
print({c: {s: (r["n_foreground"] if r else None) for s, r in stages.items()}
       for c, stages in integration_results.items()})

## 2c. CPTAC/CMI Per-Site Clinical Embedding

For each CPTAC/CMI primary site with enough samples, recompute embedding and expression-PCA UMAP+Leiden, save cluster and cross-cluster UMAPs, and run an OS Kaplan-Meier per representation. OS log-rank p-values are BH-FDR corrected across sites within each representation space, and the KM panels are redrawn with the adjusted value.

In [ ]:
# CPTAC/CMI per-primary-site clinical embedding: embedding & expression UMAP+Leiden,
# cross-cluster UMAPs, and OS Kaplan-Meier per site. OS log-rank p-values are BH-FDR
# corrected across sites within each representation space; KM panels show p(FDR).
# Sites need >= CPTAC_SITE_KM_MIN_SAMPLES samples; OS KM needs >= CPTAC_SITE_MIN_OS OS-usable.
# PDFs -> Figures/Figure4/cptac_cmi/cptac_<site>_{embedding,expression}_{umap_clusters,os_km}.pdf
CPTAC_SITE_KM_MIN_SAMPLES = 120
CPTAC_SITE_KM_MIN_GROUP = 10
CPTAC_SITE_MIN_OS = 20

CPTAC_SITE_PARAMS_DEFAULT = {
    "emb_nn": 8, "emb_min_dist": 0.3, "emb_leiden_res": 0.35,
    "expr_nn": 8, "expr_min_dist": 0.3, "expr_leiden_res": 0.35,
}
CPTAC_SITE_PARAMS = {
    # Per-site overrides, e.g. "Bronchus and lung": {"emb_leiden_res": 0.4},
}

cptac_per_site = f4.run_cptac_per_site_clinical(
    config,
    bundle=external_prediction["bundles"].get("CPTAC_CMI"),
    site_params_default=CPTAC_SITE_PARAMS_DEFAULT,
    site_params=CPTAC_SITE_PARAMS,
    site_min_samples=CPTAC_SITE_KM_MIN_SAMPLES,
    km_min_group_size=CPTAC_SITE_KM_MIN_GROUP,
    min_os_samples=CPTAC_SITE_MIN_OS,
    plot=True,
)
display(cptac_per_site["km_summary"])

## 3. External Clinical Embedding Cohort

Run one cohort at a time while tuning UMAP/Leiden parameters. Each run saves UMAP coordinates, cluster labels, KM statistics, response tests, and duration tests when requested.

In [ ]:
EXTERNAL_CLUSTER_PARAMS = {
    "IMMUNOPOG": {"embedding": {"n_neighbors": 6, "min_dist": 0.3, "leiden_resolution": 0.35}, "expression": {"n_neighbors": 5, "min_dist": 0.3, "leiden_resolution": 0.35}, "km_min_group_size": 10, "random_state": 0},
    "LIU": {"embedding": {"n_neighbors": 6, "min_dist": 0.3, "leiden_resolution": 0.35}, "expression": {"n_neighbors": 6, "min_dist": 0.3, "leiden_resolution": 0.35}, "km_min_group_size": 10, "random_state": 0},
    "RIAZ": {"embedding": {"n_neighbors": 5, "min_dist": 0.3, "leiden_resolution": 0.30}, "expression": {"n_neighbors": 4, "min_dist": 0.3, "leiden_resolution": 0.35}, "km_min_group_size": 10, "random_state": 0},
    "VAN_ALLEN": {"embedding": {"n_neighbors": 5, "min_dist": 0.3, "leiden_resolution": 0.35}, "expression": {"n_neighbors": 3, "min_dist": 0.3, "leiden_resolution": 0.25}, "km_min_group_size": 10, "random_state": 0},
    "HUGO": {"embedding": {"n_neighbors": 3, "min_dist": 0.3, "leiden_resolution": 0.50}, "expression": {"n_neighbors": 3, "min_dist": 0.3, "leiden_resolution": 0.50}, "km_min_group_size": 10, "random_state": 0},
    "METABRIC": {"embedding": {"n_neighbors": 25, "min_dist": 0.3, "leiden_resolution": 0.50}, "expression": {"n_neighbors": 25, "min_dist": 0.3, "leiden_resolution": 0.50}, "km_min_group_size": 10, "random_state": 0},
    "MORRISON": {"embedding": {"n_neighbors": 25, "min_dist": 0.3, "leiden_resolution": 0.50}, "expression": {"n_neighbors": 12, "min_dist": 0.3, "leiden_resolution": 0.50}, "km_min_group_size": 10, "random_state": 0},
}

for COHORT_TO_RUN in EXTERNAL_CLUSTER_PARAMS:
    duration_cols = []

    clinical_part = f4.run_external_clinical_cohort(
        config,
        COHORT_TO_RUN,
        params=EXTERNAL_CLUSTER_PARAMS[COHORT_TO_RUN],
        duration_cols=duration_cols,
        plot=True,
    )

    display(clinical_part["cluster_summary"])
    display(clinical_part["km_summary"])
    display(clinical_part["response_summary"])
    display(clinical_part["response_proportions"])


## 3b. Clustering-Parameter Robustness Sweep

Robustness check for the external clinical-embedding analysis: instead of a single
UMAP/Leiden setting per cohort, sweep a 5x5 grid of clustering parameters around each
cohort's chosen center (`n_neighbors` = center +/- 2, `leiden_resolution` = center +/- 0.10)
and recompute every clinical association (`km:OS`, `km:PFS`, `chisq:Response/Full_response`) in both the **embedding** and **expression** space. It reuses the
same helpers as Section 3 (`run_umap_cluster`, `expression_pca_components`, `chisq_test`,
`kruskal_test`, log-rank), so each cell is computed exactly as the main panels are, but without
plotting.

Outputs (under `Figures/Figure4/robustness_sweep/`):
- `cluster_sweep_<COHORT>.csv`, `cptac_site_sweep.csv` - per parameter-cell p-values.
- `cluster_robustness_master.csv` / `cluster_robustness_summary.csv` - per (unit, metric, space)
  roll-up: `center_p` (p at the chosen setting), `frac_sig` (fraction of the grid with p < 0.05),
  and the p-value range across the grid.

A robust separation stays significant across most of the grid (high `frac_sig`), not just at the
center. Requires `umap-learn`, `igraph`, `leidenalg`, and `lifelines`. This recomputes UMAP+Leiden
for ~650 parameter cells, so a full run takes a while; use `SWEEP_COHORTS` / `RUN_CPTAC_SWEEP` to
subset while iterating.


In [ ]:
# --- 3b. Robustness sweep: configuration + functions -------------------------------------
import numpy as np

SWEEP_OUT = config.figure_dir / "robustness_sweep"
SWEEP_OUT.mkdir(parents=True, exist_ok=True)

SIG_ALPHA = 0.05                      # significance threshold for frac_sig
NN_OFFSETS = (-2, -1, 0, 1, 2)        # n_neighbors grid = center +/- 2
RES_OFFSETS = (-0.10, -0.05, 0.0, 0.05, 0.10)   # leiden_resolution grid = center +/- 0.10

# Which units to sweep (subset while iterating; set RUN_CPTAC_SWEEP=False to skip CPTAC sites).
SWEEP_COHORTS = ["IMMUNOPOG", "LIU", "RIAZ", "VAN_ALLEN", "HUGO", "METABRIC", "MORRISON"]
RUN_CPTAC_SWEEP = True

# Center parameters = the clustering settings used for the main Figure 4 panels.
SWEEP_EXTERNAL_PARAMS = EXTERNAL_CLUSTER_PARAMS
SWEEP_CPTAC_CENTER = CPTAC_SITE_PARAMS_DEFAULT
CPTAC_SWEEP_SITE_MIN_SAMPLES = CPTAC_SITE_KM_MIN_SAMPLES
CPTAC_SWEEP_MIN_OS = CPTAC_SITE_MIN_OS
CPTAC_SWEEP_KM_MIN_GROUP = CPTAC_SITE_KM_MIN_GROUP
CPTAC_SWEEP_EXPR_PCA = 256


def _nn_grid(center):
    return sorted({max(2, int(center) + d) for d in NN_OFFSETS})


def _res_grid(center):
    return sorted({round(float(center) + d, 2) for d in RES_OFFSETS if round(float(center) + d, 2) > 0})


def _is_center(nn, res, c_nn, c_res):
    return (int(nn) == int(c_nn)) and (abs(float(res) - float(c_res)) < 1e-9)


def _km_pvalue(clinical, clusters, time_col, event_col, min_group_size):
    """Log-rank p-value for cluster labels, matching plot_km_curves' stat path (no plotting)."""
    try:
        df, _ = f4._prepare_survival_groups(
            clinical, clusters, time_col=time_col, event_col=event_col, min_group_size=int(min_group_size)
        )
        from lifelines.statistics import multivariate_logrank_test
        p = float(multivariate_logrank_test(df["time"], df["group"], df["event"]).p_value)
        return p, int(df["group"].nunique())
    except Exception:
        return np.nan, 0


def _cluster_labels(matrix, nn, res, min_dist, random_state, *, scale):
    return f4.run_umap_cluster(
        matrix, n_neighbors=int(nn), min_dist=float(min_dist), leiden_resolution=float(res),
        random_state=int(random_state), scale=scale,
    )["labels"]


def sweep_cohort(cohort):
    """5x5 (n_neighbors x leiden_res) sweep per space for one external clinical cohort."""
    params = SWEEP_EXTERNAL_PARAMS[cohort]
    rs = int(params.get("random_state", 0))
    mgs = int(params.get("km_min_group_size", 10))
    bundle = f4.load_external_bundle(f4.external_bundle_path(config, cohort))
    clinical = bundle["clinical"]
    emb = bundle["embeddings_external"]
    expression = f4.load_external_expression(bundle).loc[emb.index]
    expr_pcs = f4.expression_pca_components(expression, n_components=emb.shape[1], random_state=rs)

    # Metric selection mirrors run_external_clinical_cohort exactly.
    endpoints = [(n, t, e) for n, t, e in (("OS", "OS_time", "OS_event"), ("PFS", "PFS_time", "PFS_event"))
                 if t in clinical.columns and e in clinical.columns]
    response_cols = [c for c in ("Response", "Full_response") if c in clinical.columns]
    duration_cols = []  # Morrison OS/PFS are duration-only (no event) -> not a valid survival test

    rows = []
    for space in ("embedding", "expression"):
        c_nn = int(params[space]["n_neighbors"]); c_res = float(params[space]["leiden_resolution"])
        md = float(params[space]["min_dist"])
        source = emb if space == "embedding" else expr_pcs
        scale = space == "embedding"     # embedding is scaled, expression PCs are not (see compute_external_spaces)
        for nn in _nn_grid(c_nn):
            for res in _res_grid(c_res):
                center = _is_center(nn, res, c_nn, c_res)
                try:
                    clusters = _cluster_labels(source, nn, res, md, rs, scale=scale)
                except Exception:
                    clusters = None

                def _row(metric, p, ng):
                    rows.append({"cohort": cohort, "swept_space": space, "n_neighbors": nn,
                                 "leiden_res": res, "metric": metric, "p_value": p,
                                 "n_groups": ng, "is_center": center})

                if clusters is None:
                    for ep, _, _ in endpoints: _row(f"km:{ep}", np.nan, 0)
                    for col in response_cols: _row(f"chisq:{col}", np.nan, np.nan)
                    for col in duration_cols: _row(f"kruskal:{col}", np.nan, np.nan)
                    continue
                for ep, tcol, ecol in endpoints:
                    p, ng = _km_pvalue(clinical, clusters, tcol, ecol, mgs)
                    _row(f"km:{ep}", p, ng)
                for col in response_cols:
                    st = f4.chisq_test(clusters, clinical.loc[clusters.index, col])
                    _row(f"chisq:{col}", st.get("p_value"), np.nan)
                for col in duration_cols:
                    st = f4.kruskal_test(clusters, clinical.loc[clusters.index, col])
                    _row(f"kruskal:{col}", st.get("p_value"), st.get("n_groups"))

    df = pd.DataFrame(rows)
    df.to_csv(SWEEP_OUT / f"cluster_sweep_{cohort}.csv", index=False)
    return df


def sweep_cptac_sites():
    """Per-primary-site OS Kaplan-Meier sweep for CPTAC/CMI (km:OS only)."""
    bundle = f4.load_external_bundle(f4.external_bundle_path(config, "CPTAC_CMI"))
    clinical = bundle["clinical"]; emb = bundle["embeddings_external"]
    expression = f4.load_external_expression(bundle)
    site_col = f4.CPTAC_SITE_COL
    rs = 0
    site_counts = clinical[site_col].value_counts()
    sites = site_counts[site_counts >= CPTAC_SWEEP_SITE_MIN_SAMPLES].index.tolist()
    os_t = pd.to_numeric(clinical["OS_time"], errors="coerce")
    os_e = pd.to_numeric(clinical["OS_event"], errors="coerce")
    os_usable = os_t.notna() & os_e.notna() & (os_t > 0)

    rows = []
    for site in sites:
        label = f4._cptac_site_label(site)
        site_idx = clinical.index[clinical[site_col] == site].intersection(emb.index)
        n = len(site_idx)
        if n < 3:
            continue
        site_clin = clinical.loc[site_idx]
        n_os = int(os_usable.loc[site_idx].sum())
        emb_site = emb.loc[site_idx]
        site_pcs = f4.expression_pca_components(
            expression.loc[site_idx], n_components=min(CPTAC_SWEEP_EXPR_PCA, n - 1), random_state=rs)
        for space in ("embedding", "expression"):
            c_nn = SWEEP_CPTAC_CENTER["emb_nn" if space == "embedding" else "expr_nn"]
            c_res = SWEEP_CPTAC_CENTER["emb_leiden_res" if space == "embedding" else "expr_leiden_res"]
            md = SWEEP_CPTAC_CENTER["emb_min_dist" if space == "embedding" else "expr_min_dist"]
            source = emb_site if space == "embedding" else site_pcs
            scale = space == "embedding"
            for nn in _nn_grid(c_nn):
                for res in _res_grid(c_res):
                    center = _is_center(nn, res, c_nn, c_res)
                    if n_os < CPTAC_SWEEP_MIN_OS:
                        rows.append({"site": label, "swept_space": space, "n_neighbors": nn, "leiden_res": res,
                                     "p_value": np.nan, "n_groups": 0,
                                     "status": f"skipped: only {n_os} OS-usable", "is_center": center})
                        continue
                    try:
                        clusters = _cluster_labels(source, nn, res, md, rs, scale=scale)
                        p, ng = _km_pvalue(site_clin, clusters, "OS_time", "OS_event", CPTAC_SWEEP_KM_MIN_GROUP)
                        status = "saved" if np.isfinite(p) else "skipped: <2 groups >= min size"
                    except Exception as exc:
                        p, ng, status = np.nan, 0, f"skipped: {exc}"
                    rows.append({"site": label, "swept_space": space, "n_neighbors": nn, "leiden_res": res,
                                 "p_value": p, "n_groups": ng, "status": status, "is_center": center})
    df = pd.DataFrame(rows)
    df.to_csv(SWEEP_OUT / "cptac_site_sweep.csv", index=False)
    return df


def _rollup(detail, unit_col, metric_col):
    """Aggregate a per-cell sweep table to per (unit, metric, space) robustness statistics."""
    out = []
    for (unit, metric, space), g in detail.groupby([unit_col, metric_col, "swept_space"]):
        p = pd.to_numeric(g["p_value"], errors="coerce")
        finite = p[p.notna()]
        center = pd.to_numeric(g.loc[g["is_center"] == True, "p_value"], errors="coerce").dropna()  # noqa: E712
        out.append({
            "unit": unit, "metric": metric, "space": space,
            "center_p": float(center.iloc[0]) if len(center) else np.nan,
            "n_cells": int(len(g)), "n_valid": int(finite.shape[0]),
            "frac_sig": float((finite < SIG_ALPHA).mean()) if finite.shape[0] else np.nan,
            "p_min": float(finite.min()) if finite.shape[0] else np.nan,
            "p_median": float(finite.median()) if finite.shape[0] else np.nan,
            "p_max": float(finite.max()) if finite.shape[0] else np.nan,
        })
    return pd.DataFrame(out)


In [ ]:
# --- 3b. Robustness sweep: run + roll up -------------------------------------------------
# Runs the full sweep, writes per-cell tables and the master/summary roll-ups, and shows the
# embedding-vs-expression robustness comparison. Slow: recomputes UMAP+Leiden per grid cell.

cohort_details = []
for _cohort in SWEEP_COHORTS:
    print(f"sweeping {_cohort} ...")
    cohort_details.append(sweep_cohort(_cohort))
cohort_detail = pd.concat(cohort_details, ignore_index=True) if cohort_details else pd.DataFrame()

cptac_detail = sweep_cptac_sites() if RUN_CPTAC_SWEEP else pd.DataFrame()

# Roll-ups. Master combines cohorts and CPTAC sites; summary keeps the per-cohort view.
summary = _rollup(cohort_detail, "cohort", "metric") if not cohort_detail.empty else pd.DataFrame()

master_parts = []
if not cohort_detail.empty:
    master_parts.append(_rollup(cohort_detail, "cohort", "metric"))
if not cptac_detail.empty:
    cp = cptac_detail.copy()
    cp["unit"] = "CPTAC:" + cp["site"].astype(str)
    cp["metric"] = "km:OS"
    master_parts.append(_rollup(cp, "unit", "metric"))
master = pd.concat(master_parts, ignore_index=True) if master_parts else pd.DataFrame()

MASTER_COLS = ["unit", "metric", "space", "center_p", "n_valid", "frac_sig", "p_min", "p_max"]
SUMMARY_COLS = ["cohort", "metric", "space", "center_p", "n_cells", "n_valid", "frac_sig", "p_min", "p_median", "p_max"]

if not master.empty:
    master = master.sort_values(["unit", "metric", "space"]).reset_index(drop=True)
    master[MASTER_COLS].to_csv(SWEEP_OUT / "cluster_robustness_master.csv", index=False)
if not summary.empty:
    summary = summary.rename(columns={"unit": "cohort"}).sort_values(["cohort", "metric", "space"]).reset_index(drop=True)
    summary[SUMMARY_COLS].to_csv(SWEEP_OUT / "cluster_robustness_summary.csv", index=False)

print(f"\nWrote sweep outputs to: {SWEEP_OUT}")

# Embedding vs expression: fraction of the parameter grid that stays significant (higher = more robust).
if not master.empty:
    robustness_compare = (
        master.pivot_table(index=["unit", "metric"], columns="space", values="frac_sig")
        .rename(columns={"embedding": "frac_sig_embedding", "expression": "frac_sig_expression"})
        .reset_index()
    )
    display(robustness_compare)
    display(master[MASTER_COLS])


### 3b (cont.) Cluster-morphology supplemental figure

Beyond the p-value roll-up, visualize the clusterings themselves so the structure can be
examined directly. `plot_robustness_summary` shows `frac_sig` for the learned-embedding vs
expression-PCA space per (unit, metric); `plot_cluster_sweep_grid` renders a UMAP-panel grid
(rows = `n_neighbors`, cols = `leiden_resolution`) around each chosen center, colored by Leiden
cluster with the clinical-association p annotated per cell (green if p < 0.05) and the center
boxed. Written to `figure4/_output/robustness_sweep/`.

In [ ]:
# --- 3b. Robustness sweep: cluster-morphology supplemental figure ------------------------
# Summary: learned-embedding vs expression-PCA fraction-of-grid-significant per (unit, metric).
f4.plot_robustness_summary(master, save_path=SWEEP_OUT / "figure4_robustness_summary.pdf")


def _cohort_sweep_source(cohort, space):
    """(source, clinical, center params, scale) for a cohort + space, matching sweep_cohort."""
    p = SWEEP_EXTERNAL_PARAMS[cohort]; rs = int(p.get("random_state", 0))
    bundle = f4.load_external_bundle(f4.external_bundle_path(config, cohort))
    emb = bundle["embeddings_external"]; clinical = bundle["clinical"]
    if space == "embedding":
        return emb, clinical, p["embedding"], True
    expr = f4.load_external_expression(bundle).loc[emb.index]
    return f4.expression_pca_components(expr, n_components=emb.shape[1], random_state=rs), clinical, p["expression"], False


def _cptac_site_sweep_source(site_label, space):
    """(source, clinical, center params, scale) for one CPTAC/CMI primary site + space."""
    bundle = f4.load_external_bundle(f4.external_bundle_path(config, "CPTAC_CMI"))
    clinical = bundle["clinical"]; emb = bundle["embeddings_external"]
    site = {f4._cptac_site_label(s): s for s in clinical[f4.CPTAC_SITE_COL].dropna().unique()}[site_label]
    idx = clinical.index[clinical[f4.CPTAC_SITE_COL] == site].intersection(emb.index)
    key = "emb" if space == "embedding" else "expr"
    params = {"n_neighbors": SWEEP_CPTAC_CENTER[f"{key}_nn"],
              "leiden_resolution": SWEEP_CPTAC_CENTER[f"{key}_leiden_res"],
              "min_dist": SWEEP_CPTAC_CENTER[f"{key}_min_dist"]}
    if space == "embedding":
        return emb.loc[idx], clinical.loc[idx], params, True
    pcs = f4.expression_pca_components(f4.load_external_expression(bundle).loc[idx],
                                            n_components=min(CPTAC_SWEEP_EXPR_PCA, len(idx) - 1), random_state=0)
    return pcs, clinical.loc[idx], params, False


# Exemplar per-unit UMAP grids (embedding vs expression). Each grid recomputes ~20-25
# UMAP+Leiden fits, so keep the exemplar list short while iterating.
SWEEP_GRID_COHORTS = [("METABRIC", "km:OS"), ("MORRISON", "chisq:Response"), ("IMMUNOPOG", "km:OS")]
SWEEP_GRID_CPTAC_SITES = [("Lung", "km:OS")]

for _cohort, _metric in SWEEP_GRID_COHORTS:
    for _space in ("embedding", "expression"):
        _src, _clin, _prm, _scale = _cohort_sweep_source(_cohort, _space)
        f4.plot_cluster_sweep_grid(
            _src, _clin, center_nn=_prm["n_neighbors"], center_res=_prm["leiden_resolution"],
            min_dist=_prm["min_dist"], scale=_scale, metric=_metric,
            title=f"{_cohort} — {_space} — clustering-parameter sweep ({_metric})",
            save_path=SWEEP_OUT / f"figure4_robustness_umap_{_cohort}_{_space}.pdf")

for _site, _metric in SWEEP_GRID_CPTAC_SITES:
    for _space in ("embedding", "expression"):
        _src, _clin, _prm, _scale = _cptac_site_sweep_source(_site, _space)
        f4.plot_cluster_sweep_grid(
            _src, _clin, center_nn=_prm["n_neighbors"], center_res=_prm["leiden_resolution"],
            min_dist=_prm["min_dist"], scale=_scale, metric=_metric,
            title=f"CPTAC:{_site} — {_space} — clustering-parameter sweep ({_metric})",
            save_path=SWEEP_OUT / f"figure4_robustness_umap_CPTAC_{_site}_{_space}.pdf")
